In [ ]:
import pandas as pd
df = pd.read_csv("FINAL_ALL_FEATURES_CLEANED.csv")

In [10]:
import pandas as pd
import numpy as np
from pvlib.location import Location
from tqdm import tqdm

FILE_PATH = "FINAL_ALL_FEATURES_CLEANED.csv"
OUTPUT_FILE = "HOURLY_SOLAR_DATA_IST.csv"

PANEL_EFFICIENCY = 0.18
SYSTEM_EFFICIENCY = 0.75
AREA_PER_MW = 6500
SYSTEM_MW = 1

df = pd.read_csv(FILE_PATH)

df["solar_generated_kwh"] = (
    df["ALLSKY_SFC_SW_DWN"]
    * PANEL_EFFICIENCY
    * SYSTEM_EFFICIENCY
    * AREA_PER_MW
    * SYSTEM_MW
)

df["date"] = pd.to_datetime(
    df["YEAR"].astype(int).astype(str) + "-" +
    df["MO"].astype(int).astype(str).str.zfill(2) + "-" +
    df["DY"].astype(int).astype(str).str.zfill(2)
)

df["key"] = df["LAT"].astype(str) + "_" + df["LON"].astype(str) + "_" + df["date"].dt.strftime("%Y-%m-%d")

curve_cache = {}

def compute_curve(lat, lon, date):
    loc = Location(lat, lon, tz="Asia/Kolkata")
    times = pd.date_range(date, periods=24, freq="H", tz="Asia/Kolkata")
    solpos = loc.get_solarposition(times)
    zenith = solpos["zenith"].values

    daylight = np.where(zenith < 90, np.cos(np.radians(zenith)), 0)
    if daylight.sum() == 0:
        return np.zeros(24)
    return daylight / daylight.sum()

unique_keys = df["key"].unique()

for key in tqdm(unique_keys, desc="Computing curves"):
    lat, lon, date_str = key.split("_")
    date = pd.to_datetime(date_str)
    curve_cache[key] = compute_curve(float(lat), float(lon), date)

rows = []
for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Building hourly rows"):
    curve = curve_cache[row["key"]]
    hourly_kwh = curve * row["solar_generated_kwh"]
    hourly_allsky = curve * row["ALLSKY_SFC_SW_DWN"]

    for h in range(24):
        timestamp_ist = row["date"] + pd.Timedelta(hours=h)
        timestamp_ist = timestamp_ist.tz_localize("Asia/Kolkata")

        rows.append({
            "LAT": row["LAT"],
            "LON": row["LON"],
            "YEAR": timestamp_ist.year,
            "MONTH": timestamp_ist.month,
            "DAY": timestamp_ist.day,
            "HOUR": timestamp_ist.hour,
            "ALLSKY_SFC_SW_DWN_HOURLY": hourly_allsky[h],
            "solar_kwh": hourly_kwh[h]
        })

hourly_df = pd.DataFrame(rows)
hourly_df.to_csv(OUTPUT_FILE, index=False)

print("✅ Done! Hourly data in IST timezone saved.")


Computing curves:   0%|          | 0/54780 [00:00<?, ?it/s]C:\Users\hiten\AppData\Local\Temp\ipykernel_33064\4095874375.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  times = pd.date_range(date, periods=24, freq="H", tz="Asia/Kolkata")
Building hourly rows: 100%|██████████| 54780/54780 [01:54<00:00, 477.60it/s]


✅ Done! Hourly data in IST timezone saved.
